# MCMC — companion notebook

Runnable companion for [`mcmc.md`](./mcmc.md).

1. Random-walk Metropolis–Hastings on a 2D mixture of Gaussians (target known up to constant).
2. Trace plot, sample scatter vs. true density contours.
3. Effect of step size on acceptance rate and effective sample size.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(0)

## 1. Target: 2D mixture of two Gaussians

$\pi(\theta) \propto 0.5\, \mathcal{N}(\theta; \mu_1, I) + 0.5\, \mathcal{N}(\theta; \mu_2, I)$.

We never normalize — only log-densities up to a constant are used in the acceptance ratio.

In [ ]:
mu1 = np.array([-2.0, -2.0])
mu2 = np.array([ 2.0,  2.0])

def log_pi(theta):
    d1 = theta - mu1; d2 = theta - mu2
    # log-sum-exp of two Gaussian log-densities (ignore the (2*pi)^{-d/2} constant)
    a = -0.5 * (d1 @ d1)
    b = -0.5 * (d2 @ d2)
    m = max(a, b)
    return m + np.log(np.exp(a - m) + np.exp(b - m))

def true_density(grid):
    # For plotting only — properly normalized.
    d1 = grid - mu1; d2 = grid - mu2
    p1 = np.exp(-0.5 * np.sum(d1*d1, axis=-1)) / (2 * np.pi)
    p2 = np.exp(-0.5 * np.sum(d2*d2, axis=-1)) / (2 * np.pi)
    return 0.5 * p1 + 0.5 * p2

In [ ]:
def metropolis(log_target, x0, n_steps, step_size, seed):
    local = np.random.default_rng(seed)
    d = len(x0)
    samples = np.empty((n_steps, d))
    x = np.array(x0, dtype=float)
    logp = log_target(x)
    n_accept = 0
    for t in range(n_steps):
        proposal = x + step_size * local.standard_normal(d)
        logp_new = log_target(proposal)
        if np.log(local.random()) < logp_new - logp:
            x, logp = proposal, logp_new
            n_accept += 1
        samples[t] = x
    return samples, n_accept / n_steps

n_steps = 20_000
burn = 2_000
samples, acc = metropolis(log_pi, x0=[0.0, 0.0], n_steps=n_steps, step_size=2.0, seed=1)
print(f'acceptance rate = {acc:.3f}')
post = samples[burn:]
print(f'posterior mean   ≈ {post.mean(axis=0)}  (true mixture mean = {0.5*(mu1+mu2)})')

In [ ]:
# Trace + scatter vs. contours.
xg = np.linspace(-6, 6, 200)
yg = np.linspace(-6, 6, 200)
XX, YY = np.meshgrid(xg, yg)
grid = np.stack([XX, YY], axis=-1)
P = true_density(grid)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].plot(samples[:2000, 0], lw=0.6, label='theta_1')
axes[0].plot(samples[:2000, 1], lw=0.6, label='theta_2')
axes[0].axvline(burn, color='red', ls='--', label='end of burn-in')
axes[0].set_xlabel('step'); axes[0].set_ylabel('value')
axes[0].set_title('Trace (first 2000 steps)')
axes[0].legend()

axes[1].contour(XX, YY, P, levels=8, cmap='Reds')
axes[1].scatter(post[:, 0], post[:, 1], s=2, alpha=0.15, color='tab:blue')
axes[1].scatter(*mu1, marker='x', c='black', s=80)
axes[1].scatter(*mu2, marker='x', c='black', s=80)
axes[1].set_aspect('equal')
axes[1].set_title(f'Post-burn samples vs. true density\n(acceptance={acc:.2f})')
axes[1].set_xlabel('theta_1'); axes[1].set_ylabel('theta_2')
plt.tight_layout(); plt.show()

## 2. Step size vs. acceptance rate and effective sample size

Too small ⇒ acceptance ≈ 1, chain barely moves, high autocorrelation.
Too large ⇒ acceptance ≈ 0, chain rarely moves.
Rule of thumb: aim for acceptance ≈ 0.23–0.44.

In [ ]:
def ess_1d(x):
    """Effective sample size via initial monotone sequence estimator (Geyer)."""
    x = x - x.mean()
    n = len(x)
    var = np.dot(x, x) / n
    if var == 0:
        return float(n)
    # autocorrelation via FFT
    f = np.fft.fft(x, n=2*n)
    acf = np.fft.ifft(f * np.conjugate(f))[:n].real
    acf /= (var * np.arange(n, 0, -1))
    # sum pairs until they go negative
    tau = 1.0
    for k in range(1, n // 2):
        pair = acf[2*k - 1] + acf[2*k]
        if pair < 0:
            break
        tau += 2 * pair
    return n / tau

step_sizes = [0.1, 0.5, 1.0, 2.0, 4.0, 8.0]
rows = []
for ss in step_sizes:
    s, a = metropolis(log_pi, [0.0, 0.0], n_steps=10_000, step_size=ss, seed=7)
    post = s[1000:]
    e = 0.5 * (ess_1d(post[:, 0]) + ess_1d(post[:, 1]))
    rows.append((ss, a, e))
    print(f'step={ss:4.1f}  acceptance={a:.3f}  ESS≈{e:7.1f}')

rows = np.array(rows)
fig, ax1 = plt.subplots(figsize=(7, 4))
ax1.plot(rows[:, 0], rows[:, 1], 'o-', color='tab:blue', label='acceptance')
ax1.set_xscale('log'); ax1.set_xlabel('proposal step size')
ax1.set_ylabel('acceptance rate', color='tab:blue')
ax1.axhspan(0.234, 0.44, alpha=0.15, color='tab:blue', label='optimal band')
ax2 = ax1.twinx()
ax2.plot(rows[:, 0], rows[:, 2], 's-', color='tab:red', label='ESS')
ax2.set_ylabel('effective sample size', color='tab:red')
ax1.set_title('Step size trades off acceptance vs. mixing')
ax1.legend(loc='upper right'); plt.tight_layout(); plt.show()